# SpottingSalmon
YOLO model training script.
- To train YOLO models and evaluate output for best model.

_**!Important! Make sure you are connected to a GPU cluster i.e. salmonGPU **_

### Set up:

In [0]:
%pip install ultralytics numpy==1.26.4 --force-reinstall

In [0]:
dbutils.library.restartPython()

In [0]:
from ultralytics import YOLO
import torch
import os
import torch.distributed as dist
import mlflow

### Step 1: Set up model

In [0]:
# Name of the folder within the base directory where the labelled images sit
BASE_FOLDER = ""

# Initiate YAML file
dataset_yaml = f"/dbfs/mnt/lab/unrestricted/{BASE_FOLDER}/videos/LabelledFishYOLOv11/dataset.yaml"

yaml_content = f"""
path: /dbfs/mnt/lab/unrestricted/{BASE_FOLDER}/videos/LabelledFishYOLOv11
train: train/images
val: valid/images
nc: 1
names: ['fish']
"""

with open(dataset_yaml, "w") as f:
    f.write(yaml_content)



In [0]:
# Determine the compute device, ideally GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)  # prints the selected device for verification

# Load pre‑trained YOLOv8 model and move it to the selected device 
model = YOLO('/tmp/yolov8m.pt').to(device)

In [0]:
# Initialise distributed training (this is only useful with multiple GPU. Salmon only has 1)
#if torch.cuda.is_available():
#    dist.init_process_group(backend='nccl', init_method='env://')
 

In [0]:
# Name of the folder within the user directory where the experiment data can be stored
USER_FOLDER = ""

# Set or create an experiment
mlflow.set_experiment(f"/Users/{USER_FOLDER}/fish_yolo_experiment")


### Step 2: Train YOLOv8 model

!Note! You can use package optuna to automate hyperparameter tuning

#### Model1

In [0]:

results = model.train(
    data=dataset_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    project=f"/Users/{USER_FOLDER}/fish_yolo_experiment", # mlflow experiment path
    name="fish_yolo_m1",
    exist_ok=True,
    device=0,   
    single_cls=True
)


#### Model2
Increase batch size

In [0]:
model2 = YOLO('/tmp/yolov8m.pt').to(device)

results2 = model2.train(
    data=dataset_yaml,
    epochs=50,
    imgsz=640,
    batch=32,
    project=f"/Users/{USER_FOLDER}/fish_yolo_experiment",
    name="fish_yolo_m2",
    exist_ok=True,
    device=0,   
    single_cls=True
)


### Step 4: Train YOLOv11 model

In [0]:
# Get YOLO version 11
model_11 = YOLO('/tmp/yolo11n.pt').to(device)

In [0]:
results = model_11.train(
    data=dataset_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    project=f"/Users/{USER_FOLDER}/fish_yolo_experiment",
    name="fish_yolo_m3",
    exist_ok=True,
    device=0,   
    single_cls=True
)

#### Model11.2 
Change the batch size

In [0]:

with mlflow.start_run():
    mlflow.pytorch.log_model(model, artifact_path="model")
    results = model_11_2.train(
        data=dataset_yaml,
        epochs=50,
        imgsz=640,
        batch=32,
        project=f"/Users/{USER_FOLDER}/fish_yolo_experiment",
        name="fish_yolo_m4",
        exist_ok=True,
        device=0,   
        single_cls=True
    )

#### Model11.3
Change epochs 

In [0]:
model_11_3 = YOLO('/tmp/yolo11n.pt').to(device)

results = model_11_3.train(
    data=dataset_yaml,
    epochs=100,
    imgsz=640,
    batch=32,
    project=f"/Users/{USER_FOLDER}/fish_yolo_experiment",
    name="fish_yolo_m5",
    exist_ok=True,
    device=0,   
    single_cls=True
)

#### Model 11.4
Change batch to lower and keep epochs high

In [0]:
model_11_4 = YOLO('/tmp/yolo11n.pt').to(device)

results = model_11_4.train(
    data=dataset_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    project=f"/Users/{USER_FOLDER}/fish_yolo_experiment",
    name="fish_yolo_m6",
    exist_ok=True,
    device=0,   
    single_cls=True
)